# Distanztransformation

Jonas Frei, 16.10.2026

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.animation import FuncAnimation, PillowWriter

## Gitterkarte

In [ ]:
map_ = np.array([
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    [1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1],
    [1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 1],
    [1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1],
    [1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1],
    [1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1],
    [1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1],
    [1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1],
    [1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1],
    [1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1],
    [1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1],
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
])

## Startposition und Zielposition

In [ ]:
start = np.array([1, 1])  # [y, x]
goal = np.array([5, 7])  # [y, x]

## Visualisierung der Karte

In [ ]:
fig, ax = plt.subplots()
ax.set_aspect('equal')
ax.imshow(~map_.astype(bool), cmap='gray')
ax.set_title("Gitterkarte")
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.yaxis.set_inverted(False)
ax.plot(start[1], start[0], 'go', markersize=10, markerfacecolor='g', label="Start")
ax.plot(goal[1], goal[0], 'ro', markersize=10, markerfacecolor='r', label="Ziel")
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))

## Distanztransformation

Initialisiere Distanzen als unendlich weit entfernt

In [ ]:
D = np.full(map_.shape, np.inf)

Setze Distanz der Zielzelle auf 0

In [ ]:
D[goal[0], goal[1]] = 0

Erstelle eine Queue zum Abarbeiten aller Zellen

In [ ]:
queue = [tuple(goal)]

Definiere den Abstand zu den Nachbarzellen

In [ ]:
neighbors_offsets = [
    (-1, 0),  # Norden
    (1, 0),   # Süden
    (0, -1),  # Westen
    (0, 1),   # Osten
]

Iterative Berechnung der Distanzen

In [ ]:
frames = []

while queue:
    current = queue.pop(0)

    for dy, dx in neighbors_offsets:
        ni, nj = current[0] + dy, current[1] + dx
        if (0 <= ni < map_.shape[0] and 0 <= nj < map_.shape[1]
                and map_[ni, nj] == 0
                and D[ni, nj] > D[current[0], current[1]] + 1):
            D[ni, nj] = D[current[0], current[1]] + 1
            queue.append((ni, nj))

    frames.append(D.copy())

Vorbereiten der Colormap: Wände (höchster Wert) werden schwarz dargestellt

In [ ]:
base_cmap = plt.get_cmap('Blues', 256)
cmap_colors = base_cmap(np.linspace(0, 1, 256))
cmap_colors[-2:] = [0, 0, 0, 1]
cmap = ListedColormap(cmap_colors)

Visualisiere Endergebnis

In [ ]:
D_vis = D.copy()
D_vis[D_vis == np.inf] = np.max(D_vis[~np.isinf(D_vis)]) + 1

fig, ax = plt.subplots()
ax.set_aspect('equal')
im = ax.imshow(D_vis, cmap=cmap)
fig.colorbar(im)
ax.set_title("Distanztransformation")
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.yaxis.set_inverted(False)
ax.plot(start[1], start[0], 'go', markersize=10, markerfacecolor='g', label="Start")
ax.plot(goal[1], goal[0], 'ro', markersize=10, markerfacecolor='r', label="Ziel")
ax.legend(loc="center left", bbox_to_anchor=(1.25, 0.5))

### Animation der Distanztransformation

In [ ]:
fig_anim, ax_anim = plt.subplots(figsize=(7, 5))
ax_anim.set_aspect('equal')

vmax = np.max(frames[-1][~np.isinf(frames[-1])]) + 1
frames_vis = []
for f in frames:
    fv = f.copy()
    fv[np.isinf(fv)] = vmax
    frames_vis.append(fv)

im = ax_anim.imshow(frames_vis[0], cmap=cmap, vmin=0, vmax=vmax)
fig_anim.colorbar(im)
ax_anim.set_title("Distanztransformation (Animation)")
ax_anim.set_xlabel("X")
ax_anim.set_ylabel("Y")
ax_anim.yaxis.set_inverted(False)
ax_anim.plot(start[1], start[0], 'go', markersize=10, markerfacecolor='g', label="Start")
ax_anim.plot(goal[1], goal[0], 'ro', markersize=10, markerfacecolor='r', label="Ziel")
ax_anim.legend(loc="center left", bbox_to_anchor=(1.3, 0.5))
fig_anim.subplots_adjust(right=0.7)


def update_dt(frame):
    im.set_data(frames_vis[frame])
    return (im,)


anim_dt = FuncAnimation(fig_anim, update_dt, frames=len(frames_vis), interval=100, blit=True)
anim_dt.save('distanztransformation_animation.gif', writer=PillowWriter(fps=10))
plt.close(fig_anim)

In [ ]:
from IPython.display import Image

Image(filename='distanztransformation_animation.gif')

## Kürzesten Pfad entlang des Gradienten bestimmen

In [ ]:
path = [tuple(start)]
current = tuple(start)

Iterative Berechnung des Pfades

In [ ]:
path_frames = [list(path)]

while current != tuple(goal):
    # Evaluiere alle Nachbarzellen
    for dy, dx in neighbors_offsets:
        neighbor = (current[0] + dy, current[1] + dx)
        # Prüfe, ob Nachbarzelle innerhalb der Karte liegt und die Distanz zum Ziel kürzer ist
        if (0 <= neighbor[0] < map_.shape[0] and 0 <= neighbor[1] < map_.shape[1]
                and D[neighbor[0], neighbor[1]] < D[current[0], current[1]]):
            current = neighbor
            break
    # Füge die Zelle dem Pfad hinzu
    path.append(current)
    path_frames.append(list(path))

Visualisiere Endergebnis

In [ ]:
fig, ax = plt.subplots()
ax.set_aspect('equal')
im = ax.imshow(D_vis, cmap=cmap)
fig.colorbar(im)
ax.set_title("Kürzester Pfad zum Ziel")
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.yaxis.set_inverted(False)
ax.plot(start[1], start[0], 'go', markersize=10, markerfacecolor='g', label="Start")
ax.plot(goal[1], goal[0], 'ro', markersize=10, markerfacecolor='r', label="Ziel")
path = np.array(path)
ax.plot(path[:, 1], path[:, 0], 'c', linewidth=2, label="Pfad")
ax.yaxis.set_inverted(False)
ax.legend(loc="center left", bbox_to_anchor=(1.25, 0.5))

### Animation des Pfades

In [ ]:
fig_anim2, ax_anim2 = plt.subplots(figsize=(7, 5))
ax_anim2.set_aspect('equal')
im2 = ax_anim2.imshow(D_vis, cmap=cmap)
fig_anim2.colorbar(im2)
ax_anim2.set_title("Kürzester Pfad zum Ziel (Animation)")
ax_anim2.set_xlabel("X")
ax_anim2.set_ylabel("Y")
ax_anim2.yaxis.set_inverted(False)
ax_anim2.plot(start[1], start[0], 'go', markersize=10, markerfacecolor='g', label="Start")
ax_anim2.plot(goal[1], goal[0], 'ro', markersize=10, markerfacecolor='r', label="Ziel")
ax_anim2.legend(loc="center left", bbox_to_anchor=(1.3, 0.5))
fig_anim2.subplots_adjust(right=0.7)

path_line, = ax_anim2.plot([], [], 'c', linewidth=2)


def update_path(frame):
    p = np.array(path_frames[frame])
    path_line.set_data(p[:, 1], p[:, 0])
    return (path_line,)


anim_path = FuncAnimation(fig_anim2, update_path, frames=len(path_frames), interval=150, blit=True)
anim_path.save('distanztransformation_pfad_animation.gif', writer=PillowWriter(fps=6))
plt.close(fig_anim2)

In [ ]:
Image(filename='distanztransformation_pfad_animation.gif')